In [ ]:
# ===============================
# Imports
# ===============================

from sklearn.utils import compute_class_weight
from sklearn.metrics import classification_report
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import OneCycleLR
import pandas as pd
from transformers import RobertaTokenizer, RobertaForSequenceClassification, RobertaConfig
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import os
import shutil
import numpy as np
from collections import Counter
from torch.amp import autocast, GradScaler

# ===============================
# Device
# ===============================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.backends.cudnn.benchmark = True

# ===============================
# Load dataset
# ===============================

# DATA_PATH = "/kaggle/input/datasets/cesarwk/balanced-multiclass-sentiment-dataset-from-reddit/balanced_combined.csv"
DATA_PATH = "../Datasets/balanced_combined.csv"
rows = []
buffer = ""

with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        # combine lines (because there are multiple lines)
        buffer += " " + line

        # detect end of record
        if buffer.endswith(",0") or buffer.endswith(",1") or buffer.endswith(",2"):
            try:
                text, label = buffer.rsplit(",", 1)

                # clean text
                text = text.strip().strip('"')

                rows.append((text, int(label)))
            except Exception as e:
                pass

            buffer = ""  # reset

# create dataframe
df = pd.DataFrame(rows, columns=["text", "label"])

# basic cleaning
df = df.dropna()
df = df[df["label"].isin([0,1,2])]

print("Dataset size:", len(df))
print(df.head())
print("\nDistribución:")
print(df["label"].value_counts())



Device: cuda
Dataset size: 142549
                                                text  label
0  wanna kill want badly ive want stab front moth...      0
1  so, i''ve known this guy, let''s call him jame...      0
2  5. Don''t be too excited about your job.  This...      2
3  well students going learn go preferred methods...      1
4  Don''t let the phrase '"feel good'" and '"feel...      2

Distribución:
label
0    47530
1    47530
2    47489
Name: count, dtype: int64


In [ ]:
# ===============================
# Tokenizer
# ===============================

tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

# ===============================
# Dataset class
# ===============================

class RedditDataset(Dataset):

    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        text = str(self.texts[idx])
        label = int(self.labels[idx])

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(label, dtype=torch.long)
        }

# ===============================
# Focal Loss
# ===============================

class FocalLoss(nn.Module):

    def __init__(self, alpha=None, gamma=2.0):

        super().__init__()

        self.gamma = gamma

        if alpha is not None:
            self.alpha = torch.tensor(alpha, dtype=torch.float)
        else:
            self.alpha = None

    def forward(self, inputs, targets):

        ce_loss = torch.nn.functional.cross_entropy(inputs, targets, reduction="none")
        pt = torch.exp(-ce_loss)

        if self.alpha is not None:
            alpha_t = self.alpha.to(inputs.device)[targets]
            loss = alpha_t * ((1 - pt) ** self.gamma) * ce_loss
        else:
            loss = ((1 - pt) ** self.gamma) * ce_loss

        return loss.mean()

# ===============================
# Helper functions
# ===============================

def get_class_weights(labels, num_classes):

    counts = Counter(labels)
    total = sum(counts.values())

    weights = [total / (num_classes * counts[i]) for i in range(num_classes)]

    return torch.tensor(weights, dtype=torch.float)

def get_focal_alpha(labels, num_classes, neutral_boost=1.5):

    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.arange(num_classes),
        y=labels
    )

    alpha = np.array(class_weights, dtype=np.float32)

    alpha[1] *= neutral_boost
    alpha = alpha / alpha.sum()

    return alpha

# ===============================
# Freeze / Unfreeze
# ===============================

def freeze_roberta_layers(model, n):

    for i in range(n):
        for param in model.roberta.encoder.layer[i].parameters():
            param.requires_grad = False

    print("Frozen layers:", n)

def unfreeze_roberta_layer(model, idx):

    for param in model.roberta.encoder.layer[idx].parameters():
        param.requires_grad = True

    print("Unfroze layer:", idx)

# ===============================
# Train / Validation split
# ===============================

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["text"],
    df["label"],
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

train_dataset = RedditDataset(train_texts.tolist(), train_labels.tolist(), tokenizer)
val_dataset = RedditDataset(val_texts.tolist(), val_labels.tolist(), tokenizer)

batch_size = 24
accumulation_steps = 3

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

# ===============================
# Model
# ===============================

config = RobertaConfig.from_pretrained(
    "roberta-base",
    num_labels=3,
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1
)

model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    config=config
)

model.to(device)

# ===============================
# Loss
# ===============================

num_classes = 3

alpha = get_focal_alpha(train_labels, num_classes)

class_weights = get_class_weights(train_labels, num_classes)

focal_loss = FocalLoss(alpha=alpha)
cross_entropy = nn.CrossEntropyLoss(weight=class_weights.to(device))

use_focal = True

# ===============================
# Optimizer
# ===============================

no_decay = ['bias', 'LayerNorm.weight']

optimizer_grouped_parameters = [
    {"params": [], "weight_decay": 0.01},
    {"params": [], "weight_decay": 0.0},
]

for name, param in model.named_parameters():

    if not param.requires_grad:
        continue

    if any(nd in name for nd in no_decay):
        optimizer_grouped_parameters[1]["params"].append(param)
    else:
        optimizer_grouped_parameters[0]["params"].append(param)

optimizer = AdamW(optimizer_grouped_parameters, lr=2e-5)

# ===============================
# Scheduler
# ===============================

num_epochs = 15

lr_scheduler = OneCycleLR(
    optimizer,
    max_lr=2e-5,
    steps_per_epoch=len(train_loader)//accumulation_steps,
    epochs=num_epochs,
    pct_start=0.1,
    anneal_strategy='cos',
)

scaler = GradScaler(enabled=torch.cuda.is_available())

# ===============================
# SWA
# ===============================

from torch.optim.swa_utils import AveragedModel, SWALR, update_bn

swa_start = num_epochs - 4
swa_model = AveragedModel(model)
swa_scheduler = SWALR(optimizer, swa_lr=5e-6)

# ===============================
# Training settings
# ===============================

freeze_layers = 6
freeze_roberta_layers(model, freeze_layers)

num_roberta_layers = len(model.roberta.encoder.layer)

layers_to_unfreeze = list(
    reversed(range(num_roberta_layers - freeze_layers, num_roberta_layers))
)

current_unfreeze = 0

best_val_f1 = 0
best_state = None
patience = 2
patience_counter = 0

# ===============================
# Training loop
# ===============================

for epoch in range(num_epochs):

    print("\nEpoch", epoch+1)

    if use_focal:
        loss_fn = focal_loss
    else:
        loss_fn = cross_entropy

    model.train()

    total_loss = 0
    optimizer.zero_grad()

    loop = tqdm(enumerate(train_loader), total=len(train_loader))

    for i, batch in loop:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        with autocast(device_type="cuda" if torch.cuda.is_available() else "cpu"):

            outputs = model(input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.logits, labels) / accumulation_steps

        scaler.scale(loss).backward()

        total_loss += loss.item()

        if (i+1) % accumulation_steps == 0:

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad()

        loop.set_postfix(loss=loss.item())

    avg_train_loss = total_loss / len(train_loader)

    # ===============================
    # Validation
    # ===============================

    model.eval()

    preds = []
    labels_all = []

    with torch.no_grad():

        for batch in val_loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            with autocast(device_type="cuda" if torch.cuda.is_available() else "cpu"):

                outputs = model(input_ids, attention_mask=attention_mask)

            p = torch.argmax(outputs.logits, dim=1)

            preds.extend(p.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

    val_acc = accuracy_score(labels_all, preds)
    val_f1 = f1_score(labels_all, preds, average="weighted")

    print("Val Accuracy:", val_acc)
    print("Val F1:", val_f1)

    print(classification_report(labels_all, preds))

    # ===============================
    # SWA
    # ===============================

    if epoch >= swa_start:
        swa_model.update_parameters(model)
        swa_scheduler.step()
    else:
        lr_scheduler.step()

    # ===============================
    # Early stopping
    # ===============================

    if val_f1 > best_val_f1:

        best_val_f1 = val_f1
        best_state = model.state_dict()
        patience_counter = 0

    else:

        patience_counter += 1

        if patience_counter >= patience:

            print("Early stopping")
            break

# ===============================
# Save best model
# ===============================

model.load_state_dict(best_state)

print("Updating BN for SWA")

update_bn(train_loader, swa_model)

MODEL_DIR = "roberta_sentiment_model"

if os.path.exists(MODEL_DIR):
    shutil.rmtree(MODEL_DIR)

swa_model.module.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

print("Model saved:", MODEL_DIR)
print("Best F1:", best_val_f1)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    multilabel_confusion_matrix,
    balanced_accuracy_score,
    matthews_corrcoef
)
import seaborn as sns
import matplotlib.pyplot as plt

LABELS = ["Negative", "Neutral", "Positive"]

model.eval()

predictions = []

print("🔍 Running validation predictions...")

for text in tqdm(val_texts):

    encoding = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=256,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model(**encoding)
        probs = F.softmax(outputs.logits, dim=1)
        pred = torch.argmax(probs, dim=1).item()

    predictions.append(pred)

true_labels = val_labels.tolist()

print("✅ Predictions completed")

In [ ]:
accuracy = accuracy_score(true_labels, predictions)
f1 = f1_score(true_labels, predictions, average="weighted")
balanced_acc = balanced_accuracy_score(true_labels, predictions)
mcc = matthews_corrcoef(true_labels, predictions)

print("\n📊 Global Metrics")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1-score: {f1:.4f}")
print(f"Balanced Accuracy: {balanced_acc:.4f}")
print(f"MCC: {mcc:.4f}")

In [ ]:
conf_matrix = confusion_matrix(true_labels, predictions)

plt.figure(figsize=(6,5))
sns.heatmap(
    conf_matrix,
    annot=True,
    cmap="Blues",
    fmt="d",
    xticklabels=LABELS,
    yticklabels=LABELS
)

plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
conf_matrix_multi = multilabel_confusion_matrix(
    true_labels,
    predictions,
    labels=range(len(LABELS))
)

table_data = []

for idx, (label, matrix) in enumerate(zip(LABELS, conf_matrix_multi)):

    tn, fp, fn, tp = matrix.ravel()

    support = tp + fn

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    balanced_acc = (
        (recall + tn / (tn + fp)) / 2
        if (tn + fp) > 0
        else 0
    )

    mcc = matthews_corrcoef(
        [1 if t == idx else 0 for t in true_labels],
        [1 if p == idx else 0 for p in predictions]
    )

    table_data.append({
        "Class": label,
        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "Precision": round(precision, 3),
        "Recall": round(recall, 3),
        "F1-Score": round(f1, 3),
        "Balanced Acc.": round(balanced_acc, 3),
        "MCC": round(mcc, 3),
        "Support": support
    })

df_metrics = pd.DataFrame(table_data)

print("\n📊 Detailed Metrics Per Class")
display(df_metrics)

In [ ]:
def predict(text):

    encoding = tokenizer(
        text,
        truncation=True,
        padding=True,
        max_length=256,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model(**encoding)
        probs = F.softmax(outputs.logits, dim=1)
        pred = torch.argmax(probs).item()

    return LABELS[pred]


predict("This movie was absolutely amazing!")